# Remapeo de Detecciones: Asignación de Valor 0.5 (Detección Indirecta)

**Objetivo:** Aplicar remapeo 0.5 para beams e inclined_legs cuando el AG detecta daño en nodos cercanos pero no directamente en el elemento.

**Contexto metodológico:**
- **DeteccionOK = 1:** AG detectó daño directamente en el elemento
- **DeteccionOK = 0.5:** AG no detectó el elemento, pero SÍ detectó nodos cercanos (detección indirecta)
- **DeteccionOK = 0:** AG no detectó ni el elemento ni nodos cercanos (fallo completo)

**Pipeline de remapeo:**
```
todos_los_resultados.xlsx
   ↓
Etapa 1: Remapeo BEAMS (nodos_beams.csv)
   ↓
todos_los_resultados_remapeado_beams.xlsx
   ↓
Etapa 2: Remapeo INCLINED_LEGS (nodos_inclined_legs.csv)
   ↓
todos_los_resultados_remapeado_final.xlsx
```

**Elementos que NO se remapean:**
- `x_bracing` (mantienen valores binarios 0/1)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías cargadas")

## 1. Configuración de Rutas y Carga de Datos

In [ ]:
# Rutas absolutas
base_path = Path.home() / 'github' / 'Proyecto-doctoral'
resultados_nuevos = base_path / 'outputs' / 'resultados_nuevos'

# Selección de tipo de daño: 'abolladura' o 'corrosion'
TIPO_DANO = 'abolladura'  # Cambiar a 'corrosion' para procesar corrosión

# Rutas según tipo de daño
if TIPO_DANO == 'abolladura':
    excel_resultados = base_path / 'Resultados' / 'abolladura_2026-02-26_05-26-02' / 'todos_los_resultados.xlsx'
    salida_csvs_dir = resultados_nuevos / 'abolladuras_2026' / 'salida_csvs'
    output_dir = resultados_nuevos / 'abolladuras_2026'
    n_corridas = 1080
elif TIPO_DANO == 'corrosion':
    excel_resultados = base_path / 'Resultados' / 'corrosion_2026-02-27_06-07-17' / 'todos_los_resultados.xlsx'
    salida_csvs_dir = resultados_nuevos / 'corrosion_2026' / 'salida_csvs'
    output_dir = resultados_nuevos / 'corrosion_2026'
    n_corridas = 2160
else:
    raise ValueError(f"Tipo de daño '{TIPO_DANO}' no reconocido. Use 'abolladura' o 'corrosion'.")

# Archivos auxiliares (comunes a ambos tipos)
nodos_beams_csv = resultados_nuevos / 'nodos_beams.csv'
nodos_legs_csv = resultados_nuevos / 'nodos_inclined_legs.csv'

print(f"\n{'='*70}")
print(f"CONFIGURACIÓN - {TIPO_DANO.upper()}")
print(f"{'='*70}")
print(f"✓ Archivo de resultados: {excel_resultados.name}")
print(f"✓ Salida CSVs: {salida_csvs_dir.exists()} archivos")
print(f"✓ Nodos beams: {nodos_beams_csv.exists()}")
print(f"✓ Nodos legs: {nodos_legs_csv.exists()}")
print(f"✓ Número de corridas: {n_corridas}")

## 2. Cargar Datasets

In [ ]:
# Cargar resultados principales
print("\n📂 Cargando todos_los_resultados.xlsx...")
df_resultados = pd.read_excel(excel_resultados, engine='openpyxl')

print(f"✓ Dimensiones: {df_resultados.shape}")
print(f"✓ Columnas: {list(df_resultados.columns)}")
print(f"\n📊 Primeras 5 filas:")
display(df_resultados.head())

# Verificar tipos de elementos
if 'Tipo_elemento_a_buscar' in df_resultados.columns:
    tipos_elementos = df_resultados['Tipo_elemento_a_buscar'].value_counts()
    print(f"\n📊 Distribución de tipos de elemento:")
    print(tipos_elementos)
else:
    print("\n⚠️  Advertencia: Columna 'Tipo_elemento_a_buscar' no encontrada")
    print("   Se asumirá que todos los elementos 93-120 son beams/legs")

# Cargar mapeos de nodos
print(f"\n📂 Cargando archivos de mapeo...")
df_nodos_beams = pd.read_csv(nodos_beams_csv)
df_nodos_legs = pd.read_csv(nodos_legs_csv)

print(f"✓ Beams: {len(df_nodos_beams)} elementos mapeados")
print(f"✓ Inclined legs: {len(df_nodos_legs)} elementos mapeados")

print(f"\n📋 Ejemplo de mapeo (beams):")
display(df_nodos_beams.head())

## 3. Funciones de Remapeo

In [ ]:
def verificar_deteccion_nodos_cercanos(elemento_id, df_nodos_mapeo, csv_file):
    """
    Verifica si un elemento tiene detección en nodos cercanos.
    
    Parámetros:
    -----------
    elemento_id : int
        ID del elemento (beam o inclined_leg)
    df_nodos_mapeo : DataFrame
        DataFrame con columnas: [elemento_id, nodo_i, nodo_j, nodo_cercano_1, nodo_cercano_2]
    csv_file : Path
        Ruta al archivo CSV con detecciones nodales
    
    Retorna:
    --------
    bool : True si algún nodo cercano tiene Estado='Daño', False en caso contrario
    """
    # Obtener nodos del elemento
    col_name = 'beam' if 'beam' in df_nodos_mapeo.columns else 'inclined_leg'
    elemento_info = df_nodos_mapeo[df_nodos_mapeo[col_name] == elemento_id]
    
    if elemento_info.empty:
        return False
    
    # Obtener nodos cercanos
    nodo_cercano_1 = elemento_info['nodo_cercano_1'].values[0]
    nodo_cercano_2 = elemento_info['nodo_cercano_2'].values[0]
    
    # Cargar CSV de detecciones nodales
    try:
        df_nodos = pd.read_csv(csv_file)
        
        # Limpiar columna Estado
        df_nodos['Estado'] = df_nodos['Estado'].astype(str).str.strip()
        
        # Verificar si algún nodo cercano tiene Estado='Daño'
        nodos_con_dano = df_nodos[df_nodos['Estado'] == 'Daño']['Numero_de_nodo'].values
        
        return (nodo_cercano_1 in nodos_con_dano) or (nodo_cercano_2 in nodos_con_dano)
        
    except Exception as e:
        print(f"⚠️  Error leyendo {csv_file.name}: {e}")
        return False


def remapear_dataset(df, df_nodos_mapeo, salida_csvs_dir, tipo_elemento, n_corridas):
    """
    Aplica remapeo 0.5 a un dataset para un tipo de elemento específico.
    
    Parámetros:
    -----------
    df : DataFrame
        Dataset con resultados completos
    df_nodos_mapeo : DataFrame
        Mapeo elemento-nodos
    salida_csvs_dir : Path
        Directorio con archivos CSV de detecciones nodales
    tipo_elemento : str
        'beam' o 'inclined_leg'
    n_corridas : int
        Número total de corridas
    
    Retorna:
    --------
    DataFrame : Dataset con DeteccionOK remapeado
    dict : Estadísticas del remapeo
    """
    df_remapeado = df.copy()
    
    stats = {
        'elementos_analizados': 0,
        'elementos_remapeados': 0,
        'detecciones_antes': (df['DeteccionOK'] == 1).sum(),
        'detecciones_despues': 0
    }
    
    # Obtener lista de elementos a mapear
    col_mapeo = 'beam' if tipo_elemento == 'beam' else 'inclined_leg'
    elementos_a_mapear = df_nodos_mapeo[col_mapeo].unique()
    
    print(f"\n{'='*70}")
    print(f"REMAPEANDO: {tipo_elemento.upper()}")
    print(f"{'='*70}")
    print(f"Elementos a analizar: {len(elementos_a_mapear)}")
    print(f"Detecciones directas antes del remapeo: {stats['detecciones_antes']}\n")
    
    # Procesar cada corrida
    for i in tqdm(range(1, n_corridas + 1), desc=f"Remapeo {tipo_elemento}"):
        csv_file = salida_csvs_dir / f'ID_{i:04d}.csv'
        
        if not csv_file.exists():
            continue
        
        # Obtener fila de esta corrida
        mask_corrida = df_remapeado['ID'] == i
        
        if not mask_corrida.any():
            continue
        
        elemento_actual = df_remapeado.loc[mask_corrida, 'Elemento'].values[0]
        deteccion_actual = df_remapeado.loc[mask_corrida, 'DeteccionOK'].values[0]
        
        # Solo remapear si:
        # 1. El elemento está en la lista a mapear
        # 2. DeteccionOK == False (0)
        if elemento_actual in elementos_a_mapear and not deteccion_actual:
            stats['elementos_analizados'] += 1
            
            # Verificar detección en nodos cercanos
            tiene_deteccion_cercana = verificar_deteccion_nodos_cercanos(
                elemento_id=elemento_actual,
                df_nodos_mapeo=df_nodos_mapeo,
                csv_file=csv_file
            )
            
            if tiene_deteccion_cercana:
                df_remapeado.loc[mask_corrida, 'DeteccionOK'] = 0.5
                stats['elementos_remapeados'] += 1
    
    # Actualizar estadísticas finales
    stats['detecciones_despues'] = ((df_remapeado['DeteccionOK'] == 1) | 
                                     (df_remapeado['DeteccionOK'] == 0.5)).sum()
    
    return df_remapeado, stats

print("✓ Funciones de remapeo definidas")

## 4. Etapa 1: Remapeo de BEAMS

In [ ]:
df_remapeado_beams, stats_beams = remapear_dataset(
    df=df_resultados,
    df_nodos_mapeo=df_nodos_beams,
    salida_csvs_dir=salida_csvs_dir,
    tipo_elemento='beam',
    n_corridas=n_corridas
)

print(f"\n{'='*70}")
print("RESUMEN - REMAPEO BEAMS")
print(f"{'='*70}")
print(f"Elementos analizados: {stats_beams['elementos_analizados']}")
print(f"Elementos remapeados (0 → 0.5): {stats_beams['elementos_remapeados']}")
print(f"\nDetecciones (directas):")
print(f"  Antes: {stats_beams['detecciones_antes']}")
print(f"  Después (directas + indirectas): {stats_beams['detecciones_despues']}")
print(f"  Mejora: +{stats_beams['detecciones_despues'] - stats_beams['detecciones_antes']} detecciones")

# Guardar dataset intermedio
output_beams = output_dir / 'todos_los_resultados_remapeado_beams.xlsx'
df_remapeado_beams.to_excel(output_beams, index=False, engine='openpyxl')
print(f"\n✓ Guardado: {output_beams.name}")

## 5. Etapa 2: Remapeo de INCLINED_LEGS

In [ ]:
df_remapeado_final, stats_legs = remapear_dataset(
    df=df_remapeado_beams,  # Usar el dataset ya remapeado de beams
    df_nodos_mapeo=df_nodos_legs,
    salida_csvs_dir=salida_csvs_dir,
    tipo_elemento='inclined_leg',
    n_corridas=n_corridas
)

print(f"\n{'='*70}")
print("RESUMEN - REMAPEO INCLINED_LEGS")
print(f"{'='*70}")
print(f"Elementos analizados: {stats_legs['elementos_analizados']}")
print(f"Elementos remapeados (0 → 0.5): {stats_legs['elementos_remapeados']}")
print(f"\nDetecciones (directas + beams indirectas):")
print(f"  Antes: {stats_legs['detecciones_antes']}")
print(f"  Después (+ legs indirectas): {stats_legs['detecciones_despues']}")
print(f"  Mejora: +{stats_legs['detecciones_despues'] - stats_legs['detecciones_antes']} detecciones")

# Guardar dataset final
output_final = output_dir / 'todos_los_resultados_remapeado_final.xlsx'
df_remapeado_final.to_excel(output_final, index=False, engine='openpyxl')
print(f"\n✓ Guardado: {output_final.name}")

## 6. Resumen Final y Verificación

In [ ]:
print(f"\n{'='*70}")
print(f"RESUMEN COMPLETO - {TIPO_DANO.upper()}")
print(f"{'='*70}")

# Comparar DeteccionOK antes y después
detecciones_original = (df_resultados['DeteccionOK'] == 1).sum()
detecciones_final = ((df_remapeado_final['DeteccionOK'] == 1) | 
                     (df_remapeado_final['DeteccionOK'] == 0.5)).sum()

print(f"\n📊 Detecciones totales:")
print(f"  Original (solo directas): {detecciones_original} / {len(df_resultados)} ({100*detecciones_original/len(df_resultados):.1f}%)")
print(f"  Final (directas + indirectas): {detecciones_final} / {len(df_remapeado_final)} ({100*detecciones_final/len(df_remapeado_final):.1f}%)")
print(f"  Mejora absoluta: +{detecciones_final - detecciones_original} detecciones")
print(f"  Mejora relativa: +{100*(detecciones_final - detecciones_original)/detecciones_original:.1f}%")

# Distribución de valores de DeteccionOK
print(f"\n📊 Distribución de DeteccionOK (final):")
dist_final = df_remapeado_final['DeteccionOK'].value_counts().sort_index()
for valor, count in dist_final.items():
    if valor == 0:
        label = 'Fallo completo'
    elif valor == 0.5:
        label = 'Detección indirecta (nodos)'
    elif valor == 1:
        label = 'Detección directa'
    else:
        label = f'Valor inesperado: {valor}'
    
    print(f"  {valor}: {count} ({100*count/len(df_remapeado_final):.1f}%) - {label}")

# Archivos generados
print(f"\n📁 Archivos generados:")
print(f"  1. {output_beams.relative_to(base_path)}")
print(f"  2. {output_final.relative_to(base_path)}")

print(f"\n{'='*70}")
print("✅ Remapeo completado exitosamente")
print(f"{'='*70}")

## 7. Verificación de Casos Específicos

In [ ]:
# Mostrar ejemplos de elementos remapeados
cambios = df_remapeado_final['DeteccionOK'] != df_resultados['DeteccionOK']
df_cambios = df_remapeado_final[cambios].copy()
df_cambios['DeteccionOK_original'] = df_resultados.loc[cambios, 'DeteccionOK'].values

if len(df_cambios) > 0:
    print(f"\n📋 Ejemplos de elementos remapeados (primeros 10):")
    cols_mostrar = ['ID', 'Elemento', 'Porcentaje', 'DeteccionOK_original', 'DeteccionOK', 'N_FalsosPositivos']
    display(df_cambios[cols_mostrar].head(10))
else:
    print("\n⚠️  No se encontraron elementos remapeados (todos los elementos con detección indirecta ya tenían DeteccionOK=1)")

## 📝 Notas

**Archivos de salida:**
- `todos_los_resultados_remapeado_beams.xlsx`: Dataset con beams remapeados
- `todos_los_resultados_remapeado_final.xlsx`: Dataset final con beams + inclined_legs remapeados

**Valores de DeteccionOK después del remapeo:**
- `0`: Fallo completo (AG no detectó ni elemento ni nodos cercanos)
- `0.5`: Detección indirecta (AG detectó nodos cercanos pero no el elemento directamente)
- `1`: Detección directa (AG detectó el elemento correctamente)

**Próximos pasos:**
1. Repetir proceso para corrosión (cambiar `TIPO_DANO = 'corrosion'` en celda 2)
2. Calcular ICD con datos remapeados
3. Comparar ICD antiguo vs nuevo
4. Análisis estadístico de vectores alpha